# ProofHound — Evaluate Prompt v4 on all benchmark cases

Runs the 7B v3-2048 adapter with **Prompt v4** (static call-chain injection) on all 35 benchmark cases.
Prints a running summary every 5 cases so you can follow along.

**Requirements (local):** Docker Desktop running · GPU available · venv active (`source .venv/Scripts/activate`)

Results go to `runs/compare-v4-full/`. The notebook is safe to re-run; it stops the old server first.

In [ ]:
import subprocess, sys, time, os, json, textwrap
from pathlib import Path

ROOT   = Path(".").resolve()   # assumes notebook launched from repo root
PY     = ROOT / ".venv/Scripts/python"
ADAPTER = ROOT / "models/harness-adapter-7b-v3-2048"
BASE   = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"
PORT   = 8000
OUT    = ROOT / "runs/compare-v4-full"

assert PY.exists(),     f"venv not found at {PY}"
assert ADAPTER.exists(), f"adapter not found at {ADAPTER}"
print("root :", ROOT)
print("adapter :", ADAPTER.name)
print("out dir :", OUT)

In [ ]:
# ── Kill any leftover server on port 8000 ──────────────────────────────────────
import socket, signal

def _port_pid(port):
    try:
        r = subprocess.run(
            ["netstat", "-ano"], capture_output=True, text=True
        )
        for line in r.stdout.splitlines():
            if f":{port}" in line and "LISTENING" in line:
                return int(line.strip().split()[-1])
    except Exception:
        pass
    return None

def stop_server():
    pid = _port_pid(PORT)
    if pid:
        print(f"Killing existing server (PID {pid})")
        subprocess.run(["taskkill", "/F", "/T", "/PID", str(pid)],
                       capture_output=True)
        for _ in range(30):
            time.sleep(1)
            if not _port_pid(PORT):
                break
        print("Server stopped.")
    else:
        print("No server running on port", PORT)

stop_server()

In [ ]:
# ── Start the 7B inference server ─────────────────────────────────────────────
import urllib.request

serve_log = OUT.parent / "serve-v4-full.log"
OUT.parent.mkdir(parents=True, exist_ok=True)

env = {**os.environ, "HF_HUB_OFFLINE": "1", "PYTHONUTF8": "1"}
srv = subprocess.Popen(
    [str(PY), "training/serve/serve_local.py",
     "--adapter", str(ADAPTER),
     "--base",    BASE,
     "--device",  "cuda",
     "--load-4bit",
     "--port",    str(PORT)],
    cwd=str(ROOT), env=env,
    stdout=open(serve_log, "w"), stderr=subprocess.STDOUT,
)

print(f"Server PID {srv.pid} — waiting for it to be ready (up to 5 min) ...")
ready = False
for i in range(100):
    time.sleep(3)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=2)
        ready = True
        break
    except Exception:
        if i % 10 == 9:
            print(f"  still loading ({(i+1)*3}s) ...")

if not ready:
    print("Server did not start — check", serve_log)
    srv.kill()
    raise RuntimeError("Server startup failed")
print(f"Server ready after {(i+1)*3}s")

In [ ]:
# ── Run the evaluation — all cases, progress every 5 ─────────────────────────
compare_log = ROOT / "runs" / "compare-v4-full.log"

proc = subprocess.Popen(
    [str(PY), "-m", "proofhound.evaluation.compare",
     "--models",         "local-harness",
     "--out",            str(OUT),
     "--progress-every", "5"],
    cwd=str(ROOT),
    env={**os.environ, "PYTHONUTF8": "1"},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

with open(compare_log, "w", encoding="utf-8") as log_f:
    for line in proc.stdout:
        print(line, end="", flush=True)
        log_f.write(line)

proc.wait()
print(f"\nCompare exited with code {proc.returncode}")

In [ ]:
# ── Stop the server ────────────────────────────────────────────────────────────
stop_server()

In [ ]:
# ── Display final results ──────────────────────────────────────────────────────
summary = OUT / "summary.json"
if not summary.exists():
    print("summary.json not found — run did not complete")
else:
    data = json.loads(summary.read_text(encoding="utf-8"))
    models = data["models"]
    by_class = data["by_class"]

    def pct_bar(label, n, total, width=30):
        pct = n / max(total, 1)
        filled = int(pct * width)
        bar = "█" * filled + "░" * (width - filled)
        return f"{label:<12} |{bar}| {n}/{total} ({100*pct:.0f}%)"

    for row in models:
        model = row["model"]
        n = row["findings"]
        print(f"\n{'='*55}")
        print(f"  {model}  —  {n} findings,  {row['errors']} errors")
        print(f"{'='*55}")
        for stage in ("syntax_ok", "loaded", "reached", "confirmed"):
            val = row[stage]
            count = int(val.split()[0]) if isinstance(val, str) else val
            print(pct_bar(stage, count, n))
        print(f"  avg {row['avg_tokens']:,} tokens / {row['avg_seconds']}s per finding")

    print("\n── by vuln class ──")
    for row in by_class:
        n = row["findings"]
        reached_val = row["reached"]
        reached_n = int(reached_val.split()[0]) if isinstance(reached_val, str) else reached_val
        confirmed_val = row["confirmed"]
        confirmed_n = int(confirmed_val.split()[0]) if isinstance(confirmed_val, str) else confirmed_val
        print(f"  {row['vuln_class']:<25} reached {reached_n}/{n}  confirmed {confirmed_n}/{n}")

In [ ]:
# ── Per-finding detail ─────────────────────────────────────────────────────────
STAGES = ("syntax_ok", "loaded", "reached", "confirmed")
per = data["per_finding"]
print(f"{'case':<35} {'class':<22} {'best stage'}")
print("-" * 75)
for r in per:
    best = next((s for s in reversed(STAGES) if r.get(s)), "failed")
    print(f"{r['case']:<35} {r['vuln_class']:<22} {best}")